# DPO from scratch: Google Colab T4 runner

> **Before opening this notebook in Colab:** commit and push your current local changes to the repository branch you plan to run. Colab clones GitHub and cannot see unpushed files. From the local project directory, inspect `git status --short`, then stage the intended project files (for example `git add src configs docs scripts tests notebooks README.md requirements.txt`), inspect `git diff --cached --check`, commit, and run `git push origin main` (or push your chosen branch). Do not stage secret files or place credentials in this notebook/repository. Then set `BRANCH` below to that branch.

This notebook starts with a 128-example / 512-token smoke run. It performs SFT once, then DPO from that exact SFT checkpoint. After the smoke run succeeds, increase the configured sample cap and rerun SFT before DPO. The notebook has not been validated on an actual Colab T4.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/Adhilmohdkt/dpo-from-scratch.git'
BRANCH = 'main'  # Change this to the branch you just pushed.
PROJECT_DIR = Path('/content/dpo-from-scratch')
if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_DIR)], check=True)
%cd /content/dpo-from-scratch

## Editable run settings

The initial values are a smoke test. Keep `MAX_TRAIN_SAMPLES = 128` and `MAX_LENGTH = 512` until both stages and evaluation finish successfully. Then increase the cap (or set it to `None` for the full training split) and sequence length as your runtime permits.

In [ ]:
# Model, data, compute, LoRA, beta, seed, and artifact paths are configurable here.
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
DATASET_ID = 'HuggingFaceH4/ultrafeedback_binarized'
RUN_TAG = 'smoke-128x512'  # Change to a new tag when starting a larger run.
MAX_TRAIN_SAMPLES = 128  # Smoke run; set None after success for the full train split.
MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
SEED = 42
SMOKE_BETA = 0.1
RUN_BETA_SWEEP = False  # Turn on only after the SFT + one-beta smoke run succeeds.
BETA_VALUES = [0.05, 0.1, 0.5]
RUN_SELECTION_EXPERIMENT = False  # Optional, after core runs work.
SELECTION_SUBSET_SIZE = 32
SELECTION_BETA = 0.1
FINAL_DPO_LABEL = 'beta-0.1'  # Used only when EVAL_SPLIT='test'.
REQUIRE_GPU = True
RESUME_SFT_CHECKPOINT = None  # Example: a checkpoint-N directory beneath SFT_PATH.
RESUME_DPO_CHECKPOINTS = {}  # Map run label to checkpoint-N path when resuming.
DRIVE_ROOT = Path('/content/drive/MyDrive/dpo-from-scratch')
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'
REFERENCE_ROOT = DRIVE_ROOT / 'reference_logps'
LOG_ROOT = DRIVE_ROOT / 'logs'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
SFT_PATH = CHECKPOINT_ROOT / RUN_TAG / 'sft'
EVAL_SPLIT = 'validation'  # Use 'test' only once after model/beta selection is complete.

In [ ]:
import subprocess, sys, shutil
gpu_cmd = shutil.which('nvidia-smi')
gpu = subprocess.run([gpu_cmd, '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if gpu_cmd else None
if gpu and gpu.returncode == 0 and gpu.stdout.strip():
    print('Visible GPU:', gpu.stdout.strip().splitlines()[0])
elif REQUIRE_GPU:
    raise RuntimeError('GPU training was requested, but no GPU is visible. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, and rerun.')
else:
    print('No GPU detected; continuing only because REQUIRE_GPU=False.')

# PyTorch 2.5.1 has a CUDA 12.4 wheel suitable for this pinned environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
from google.colab import drive
drive.mount('/content/drive')
for directory in (CHECKPOINT_ROOT, REFERENCE_ROOT, LOG_ROOT, OUTPUT_ROOT, DRIVE_ROOT / 'configs', DRIVE_ROOT / 'selections'):
    directory.mkdir(parents=True, exist_ok=True)
import os
os.environ['HF_HOME'] = str(DRIVE_ROOT / 'hf_cache')
os.environ['HF_HUB_CACHE'] = str(DRIVE_ROOT / 'hf_cache' / 'hub')
os.environ['HF_DATASETS_CACHE'] = str(DRIVE_ROOT / 'hf_cache' / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(DRIVE_ROOT / 'hf_cache' / 'hub')
import torch, transformers, peft, datasets, trl, accelerate, yaml
print({'torch': torch.__version__, 'transformers': transformers.__version__, 'peft': peft.__version__, 'datasets': datasets.__version__, 'trl': trl.__version__, 'accelerate': accelerate.__version__})
if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError('CUDA became unavailable after dependency installation; select a T4 runtime and rerun.')
if torch.cuda.is_available(): print('Verified GPU:', torch.cuda.get_device_name(0))
print('Artifacts and model cache:', DRIVE_ROOT)

In [ ]:
import copy, yaml, json, sys
from datetime import datetime, timezone
with open(PROJECT_DIR / 'configs/default.yaml', encoding='utf-8') as stream:
    BASE_CONFIG = yaml.safe_load(stream)
BASE_CONFIG['seed'] = SEED
BASE_CONFIG['model']['name_or_path'] = MODEL_ID
BASE_CONFIG['model']['torch_dtype'] = 'float16'
BASE_CONFIG['model']['lora'].update({'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'target_modules': LORA_TARGET_MODULES})
BASE_CONFIG['data'].update({'name': DATASET_ID, 'max_train_samples': MAX_TRAIN_SAMPLES, 'max_length': MAX_LENGTH, 'train_row_ids_file': None})
BASE_CONFIG['training'].update({'batch_size': BATCH_SIZE, 'gradient_accumulation_steps': GRAD_ACCUMULATION, 'device': 'cuda', 'output_dir': str(SFT_PATH), 'resume_from_checkpoint': RESUME_SFT_CHECKPOINT})
BASE_CONFIG['dpo'].update({'sft_checkpoint': str(SFT_PATH), 'reference_logps_path': str(REFERENCE_ROOT / f'full-{RUN_TAG}-seed-{SEED}.pt')})
BASE_CONFIG['evaluation'].update({'split': EVAL_SPLIT, 'batch_size': BATCH_SIZE, 'max_new_tokens': 128})
def save_config(config, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
    return path
def run_logged(args, log_path):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print('Running:', ' '.join(map(str, args)))
    with log_path.open('a', encoding='utf-8') as log:
        result = subprocess.run(args, cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    if result.returncode: raise RuntimeError(f'Command failed ({result.returncode}); inspect Drive log: {log_path}')
    print('Log:', log_path)
print('Smoke config:', {'model': MODEL_ID, 'examples': MAX_TRAIN_SAMPLES, 'max_length': MAX_LENGTH, 'seed': SEED})

## Stage 1: SFT

Run SFT once. A disconnected session can resume from a periodic `checkpoint-N` on Drive by setting `RESUME_SFT_CHECKPOINT` above and rerunning this cell. Checkpoint resumption requires that the model/data/training settings stay the same.

In [ ]:
SFT_CONFIG_PATH = save_config(BASE_CONFIG, DRIVE_ROOT / 'configs' / f'sft-seed-{SEED}.yaml')
run_logged([sys.executable, '-m', 'src.train_sft', '--config', str(SFT_CONFIG_PATH)], LOG_ROOT / f'sft-seed-{SEED}.log')
assert (SFT_PATH / 'adapter_config.json').exists(), 'SFT did not produce the expected adapter'
assert (SFT_PATH / 'sft_manifest.json').exists(), 'SFT did not produce its exact-base manifest'

## Stage 2: DPO and beta sweep

The first pass runs one beta (`SMOKE_BETA`). After it succeeds, set `RUN_BETA_SWEEP = True` and rerun this cell for 0.05, 0.1, and 0.5. Each run loads the same SFT adapter and same dataset split/seed; each beta has its own output and reference-score cache path. To resume, populate `RESUME_DPO_CHECKPOINTS` with that run's checkpoint path.

In [ ]:
active_betas = BETA_VALUES if RUN_BETA_SWEEP else [SMOKE_BETA]
DPO_CONFIGS = {}
for beta in active_betas:
    label = f'beta-{beta:g}'
    config = copy.deepcopy(BASE_CONFIG)
    config['dpo'].update({'beta': float(beta), 'output_dir': str(CHECKPOINT_ROOT / RUN_TAG / 'dpo' / f'seed-{SEED}' / label), 'reference_logps_path': str(REFERENCE_ROOT / f'full-{RUN_TAG}-seed-{SEED}.pt')})
    config['training']['resume_from_checkpoint'] = None
    config_path = save_config(config, DRIVE_ROOT / 'configs' / f'dpo-{label}-seed-{SEED}.yaml')
    command = [sys.executable, '-m', 'src.train_dpo', '--config', str(config_path)]
    if RESUME_DPO_CHECKPOINTS.get(label): command.extend(['--resume', RESUME_DPO_CHECKPOINTS[label]])
    run_logged(command, LOG_ROOT / f'dpo-{label}-seed-{SEED}.log')
    DPO_CONFIGS[label] = (config, config_path)
print('Completed beta runs:', list(DPO_CONFIGS))

## Scale after the smoke run

After reviewing the SFT/DPO logs and validating the adapters, increase `MAX_TRAIN_SAMPLES` (or set it to `None` for the complete training split) and, if memory permits, increase `MAX_LENGTH`. Change `RUN_TAG` (for example to `full-data-1024`) so the larger run gets separate checkpoints/caches; rerun SFT to create its reference checkpoint, then rerun DPO. Keep validation for model and beta selection. The final `test_prefs` split should be run only after choices are frozen. Preserve the seed and record each run separately when comparing seeds. If T4 memory is insufficient, reduce sequence length before increasing batch size.

## Optional research add-on: preference-data selection

This is a simplified experiment inspired by [Less is More](https://arxiv.org/abs/2502.14560), not a reproduction. It selects only from the training candidate pool after the deterministic validation partition and sample cap. Its pre-DPO external margin is `score_chosen - score_rejected` from UltraFeedback; the selected arm takes the largest values. The random arm samples the same number of rows without replacement using `SEED`; the full arm uses all candidate rows. No pilot DPO model or held-out examples are used for selection. Keep the same SFT checkpoint, beta, seed, epochs, and optimization settings across arms.

In [ ]:
if RUN_SELECTION_EXPERIMENT:
    selection_root = DRIVE_ROOT / 'selections' / f'seed-{SEED}'
    selection_cfg = save_config(BASE_CONFIG, DRIVE_ROOT / 'configs' / f'selection-seed-{SEED}.yaml')
    run_logged([sys.executable, 'scripts/select_preference_data.py', '--config', str(selection_cfg), '--subset-size', str(SELECTION_SUBSET_SIZE), '--output-dir', str(selection_root)], LOG_ROOT / f'selection-seed-{SEED}.log')
    arms = {'margin': str(selection_root / 'margin_subset.json'), 'random': str(selection_root / 'random_subset.json'), 'full': None}
    selection_beta = SELECTION_BETA
    for arm, ids_file in arms.items():
        config = copy.deepcopy(BASE_CONFIG)
        config['dpo'].update({'beta': selection_beta, 'output_dir': str(CHECKPOINT_ROOT / RUN_TAG / 'selection' / f'seed-{SEED}' / arm), 'reference_logps_path': str(REFERENCE_ROOT / f'selection-{RUN_TAG}-{arm}-seed-{SEED}.pt')})
        config['data']['train_row_ids_file'] = ids_file
        arm_config_path = save_config(config, DRIVE_ROOT / 'configs' / f'selection-{arm}-seed-{SEED}.yaml')
        run_logged([sys.executable, '-m', 'src.train_dpo', '--config', str(arm_config_path)], LOG_ROOT / f'selection-{arm}-seed-{SEED}.log')
    selection_eval = copy.deepcopy(BASE_CONFIG)
    selection_eval['dpo']['sft_checkpoint'] = str(SFT_PATH)
    selection_eval['evaluation'].update({'split': 'validation', 'output_dir': str(OUTPUT_ROOT / 'selection' / f'seed-{SEED}')})
    selection_eval['evaluation']['checkpoints'] = {'base': None, 'sft': str(SFT_PATH)}
    selection_eval['evaluation']['beta_by_checkpoint'] = {}
    for arm in arms:
        label = f'selection_{arm}'
        selection_eval['evaluation']['checkpoints'][label] = str(CHECKPOINT_ROOT / RUN_TAG / 'selection' / f'seed-{SEED}' / arm)
        selection_eval['evaluation']['beta_by_checkpoint'][label] = selection_beta
    selection_eval_path = save_config(selection_eval, DRIVE_ROOT / 'configs' / f'selection-eval-seed-{SEED}.yaml')
    run_logged([sys.executable, '-m', 'src.evaluate', '--config', str(selection_eval_path), '--split', 'validation'], LOG_ROOT / f'selection-eval-seed-{SEED}.log')
    print('Selection arms trained:', list(arms))
else:
    print('Selection experiment disabled; set RUN_SELECTION_EXPERIMENT=True after the core smoke run.')

## Evaluation and response-length CSVs

Base, SFT, and DPO use the same deterministic generation settings and prompts for each evaluation invocation. The evaluator writes per-example `*_response_lengths.csv` plus metric CSV/JSON. Length is descriptive only; longer responses are not necessarily better. Evaluate on validation while selecting a model/beta. After final choices are fixed, rerun with `EVAL_SPLIT = 'test'` and only the selected DPO checkpoint.

In [ ]:
eval_config = copy.deepcopy(BASE_CONFIG)
eval_config['dpo']['sft_checkpoint'] = str(SFT_PATH)
eval_config['evaluation']['output_dir'] = str(OUTPUT_ROOT / f'seed-{SEED}')
eval_config['evaluation']['checkpoints'] = {'base': None, 'sft': str(SFT_PATH)}
eval_config['evaluation']['beta_by_checkpoint'] = {}
labels_to_eval = [FINAL_DPO_LABEL] if EVAL_SPLIT == 'test' else list(DPO_CONFIGS)
for label in labels_to_eval:
    config, _ = DPO_CONFIGS[label]
    eval_label = label.replace('-', '_')
    eval_config['evaluation']['checkpoints'][eval_label] = config['dpo']['output_dir']
    eval_config['evaluation']['beta_by_checkpoint'][eval_label] = config['dpo']['beta']
eval_config_path = save_config(eval_config, DRIVE_ROOT / 'configs' / f'eval-seed-{SEED}.yaml')
run_logged([sys.executable, '-m', 'src.evaluate', '--config', str(eval_config_path), '--split', EVAL_SPLIT], LOG_ROOT / f'eval-{EVAL_SPLIT}-seed-{SEED}.log')

In [ ]:
run_logged([sys.executable, 'scripts/aggregate_results.py', '--root', str(OUTPUT_ROOT), '--output', str(OUTPUT_ROOT / 'summary_metrics.csv')], LOG_ROOT / 'aggregate.log')
print('Summary CSV:', OUTPUT_ROOT / 'summary_metrics.csv')